# Opdracht 4 · Decision tree
### Classificatie-training · Antwoordmodel-versie

Decision trees zijn de meest interpreteerbare modellen: je kunt letterlijk de beslisregels zien.
In deze opdracht ga je:

1. Een **decision tree** trainen op Iris
2. De boom **visualiseren** met `plot_tree`
3. De **eerste twee splitsingen** voorlezen (“wat vraagt het model eerst?”)
4. De **boomdiepte** variëren en het overfitting-effect zien
5. Een plot maken van train- vs. test-accuracy per diepte


## 0. Voorbereiding

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")
print("Bibliotheken geladen.")

In [ ]:
# Data laden en splitsen
data = load_iris(as_frame=True)
df = data.frame.copy()
df["soort"] = df["target"].map({0:"setosa",1:"versicolor",2:"virginica"})
y = df["soort"]; X = df[data.feature_names]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

## 1. Een decision tree trainen

### TODO 1 — Train een decision tree en meet de accuracy
Gebruik `DecisionTreeClassifier(random_state=42)` (nog zonder diepte-beperking).

In [ ]:
# ANTWOORD
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
acc = accuracy_score(y_test, tree.predict(X_test))
print(f"Decision tree accuracy: {acc:.3f}")

## 2. De boom visualiseren
De kracht van een decision tree: je ziet letterlijk de beslisregels. We trainen een boom met
maximale diepte 3 (anders wordt het plaatje te groot om te lezen).

In [ ]:
# Een kleinere boom voor de visualisatie (al ingevuld)
tree_klein = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_klein.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(14, 7))
plot_tree(tree_klein,
          feature_names=data.feature_names,
          class_names=tree_klein.classes_,
          filled=True, rounded=True, fontsize=10, ax=ax)
plt.title("Decision tree (max_depth=3)")
plt.show()

### TODO 2 — Lees de eerste twee splitsingen voor
Bekijk de boom hierboven en beschrijf in je eigen woorden:
- Welke vraag stelt het model **eerst** (de wortelknoop)?
- Welke vraag stelt het model **als tweede** (in de tak die niet meteen tot een klasse leidt)?

Vul je antwoord in als string variabelen hieronder.

In [ ]:
# ANTWOORD
eerste_split = "petal length (cm) <= 2.45  -> als ja, dan setosa"
tweede_split = "petal width (cm) <= 1.75   -> verdere splitsing tussen versicolor en virginica"

print("Eerste split: ", eerste_split)
print("Tweede split: ", tweede_split)

## 3. Boomdiepte variëren
Een te diepe boom kent de trainingsdata uit z'n hoofd (overfitting). We vergelijken train- en
test-accuracy bij verschillende diepten.

### TODO 3 — Train bomen van verschillende diepte
Voor elke diepte in `dieptes`: train een tree, bereken train-accuracy en test-accuracy,
en sla op in `resultaten`.

In [ ]:
# ANTWOORD
dieptes = [1, 2, 3, 5, None]
resultaten = []
for d in dieptes:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    train_acc = t.score(X_train, y_train)
    test_acc = t.score(X_test, y_test)
    resultaten.append({"diepte": str(d), "train_acc": train_acc, "test_acc": test_acc})

resultaten = pd.DataFrame(resultaten)
print(resultaten.round(3))

## 4. Plot van train- vs. test-accuracy
De plot is al voor je gemaakt — voer hem uit.

In [ ]:
# Plot (al ingevuld)
plt.figure(figsize=(7, 4))
x = range(len(resultaten))
plt.plot(x, resultaten["train_acc"], marker="o", color="#4473C5", label="train accuracy")
plt.plot(x, resultaten["test_acc"], marker="s", color="#D97733", label="test accuracy")
plt.xticks(x, resultaten["diepte"])
plt.xlabel("Boomdiepte"); plt.ylabel("Accuracy")
plt.title("Train vs. test accuracy per diepte")
plt.legend(); plt.ylim(0, 1.05); plt.show()

> **Let op het patroon:** train-accuracy stijgt richting 1.0 als de boom dieper wordt.
> Test-accuracy stijgt eerst mee, maar daarna kan hij weer dalen — dat gat is overfitting.

## 5. Bespreking
Bespreek met je buur:

- Welke vraag stelt de boom als eerste, en is die keuze logisch gegeven de pairplot uit opdracht 1?
- Bij welke diepte presteert de boom het best op de test-set?
- Wat zie je gebeuren met train-accuracy als de boom dieper wordt? En met test-accuracy?
- Wat is het voordeel van een decision tree boven logistische regressie? En het nadeel?

---
### Toelichting voor de trainer
- De **eerste split** is altijd op `petal length` (rond 2.45) — die scheidt setosa perfect af.
  De **tweede split** is op `petal width` (rond 1.75) en scheidt versicolor van virginica.
- **Depth sweep:** depth 3 is meestal het sweet spot (test ≈ 0.97). Bij None krijg je train=1.0
  en test ≈ 0.93 — klassiek overfitting-patroon.
- Decision trees zijn extra waardevol in gesprekken met stakeholders: “Het model zegt setosa
  omdat petal length kleiner is dan 2.45” is direct uitlegbaar.